# ST-GCN Study

Systematic comparison of ST-GCN (Spatial-Temporal Graph Convolutional Network) configurations
for per-rep push-up form classification.

**Key advantage over R3D:** ST-GCN operates on skeleton keypoints (joint coordinates),
not raw pixels. This makes it invariant to appearance, background, lighting, and camera angle.

**Study design:**
1. **Experiment A**: Baseline ST-GCN (medium, 2-channel xy)
2. **Experiment B**: Input channels comparison (2-channel xy vs 3-channel xy+confidence)
3. **Experiment C**: Model capacity (small vs medium vs large)
4. **Experiment D**: Temporal sampling (16 vs 32 vs 64 frames)
5. **Experiment E**: Hyperparameter tuning (LR + batch size)
6. **Overfitting analysis** + final comparison

All experiments use 5-fold stratified CV split by video to prevent data leakage.

## Setup + Data Loading

In [ ]:
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import PushUpSTGCN, CONFIGS
from datasets import PushUpRepSkeletonDataset
from training import run_rep_kfold_cv
from data_loader import load_annotations, extract_keypoints, attach_keypoints

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================================================
# CONFIGURATION — Vast.ai paths
# ============================================================
ANNOTATIONS   = Path("annotations_template.xlsx")
VIDEO_DIR     = Path("videos")
KEYPOINT_DIR  = Path("keypoints")
OUTPUT_DIR    = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CV settings
RANDOM_STATE = 42
N_SPLITS     = 5

# Device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")
print(f"Video dir: {VIDEO_DIR}")
print(f"Annotations: {ANNOTATIONS}")

In [ ]:
# Load annotations and attach keypoints
rep_segments = load_annotations(ANNOTATIONS, VIDEO_DIR)

n_good = sum(1 for r in rep_segments if r["label"] == 0)
n_bad  = sum(1 for r in rep_segments if r["label"] == 1)
unique_videos = set(r["video_id"] for r in rep_segments)

print(f"Total reps: {len(rep_segments)} (good={n_good}, bad={n_bad})")
print(f"Unique videos: {len(unique_videos)}")
print(f"Mean reps/video: {len(rep_segments)/max(len(unique_videos),1):.1f}")

if rep_segments:
    rep_lengths = [r["end_frame"] - r["start_frame"] for r in rep_segments]
    print(f"Rep length (frames): min={min(rep_lengths)}, max={max(rep_lengths)}, "
          f"median={np.median(rep_lengths):.0f}")

In [ ]:
# Extract keypoints (skips already-extracted videos)
extract_keypoints(rep_segments, KEYPOINT_DIR)

# Attach keypoints to rep dicts
attach_keypoints(rep_segments, KEYPOINT_DIR)

# Verify
has_kps = sum(1 for r in rep_segments if "keypoints" in r)
print(f"Reps with keypoints: {has_kps}/{len(rep_segments)}")

# Filter to only reps with keypoints
reps_with_kps = [r for r in rep_segments if "keypoints" in r]
print(f"Using {len(reps_with_kps)} reps for experiments")

---
## Experiment A: Baseline ST-GCN

Medium architecture (3 blocks: 64-64-128), 2-channel input (x,y), 32 frames per rep.
This is the default configuration.

In [ ]:
# Baseline: medium, 2-channel, 32 frames
BASELINE_N_FRAMES = 32
BASELINE_IN_CH = 2
BASELINE_CHANNELS = CONFIGS["medium"]  # [64, 64, 128]

def make_baseline_model():
    return PushUpSTGCN(in_channels=BASELINE_IN_CH, channels=BASELINE_CHANNELS)

def make_baseline_dataset(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                    in_channels=BASELINE_IN_CH, augment=False)

def make_baseline_dataset_aug(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                    in_channels=BASELINE_IN_CH, augment=True)

m = make_baseline_model()
print(f"Experiment A: Baseline ST-GCN")
print(f"  Architecture: {BASELINE_CHANNELS}")
print(f"  Input: {BASELINE_IN_CH} channels, {BASELINE_N_FRAMES} frames")
print(f"  Trainable params: {m.trainable_param_count():,}")

t0 = time.time()
results_A = run_rep_kfold_cv(
    model_factory=make_baseline_model,
    dataset_factory=make_baseline_dataset,
    rep_segments=reps_with_kps,
    n_splits=N_SPLITS,
    n_epochs=50,
    batch_size=16,
    lr=1e-3,
    patience=15,
    device_str=DEVICE,
    random_state=RANDOM_STATE,
    train_dataset_factory=make_baseline_dataset_aug,
)
time_A = time.time() - t0

accs_A = [f["val_accuracy"] for f in results_A["fold_results"]]
print(f"\nExperiment A results:")
print(f"  Per-fold: {[f'{a:.1%}' for a in accs_A]}")
print(f"  Mean: {np.mean(accs_A):.1%} +/- {np.std(accs_A):.1%}")
print(f"  Time: {time_A:.0f}s")

---
## Experiment B: Input Channels (2 vs 3)

Compare 2-channel (x, y) vs 3-channel (x, y, confidence) input.
The confidence channel tells the model how reliable each keypoint is.

In [ ]:
# 3-channel: x, y, confidence
def make_3ch_model():
    return PushUpSTGCN(in_channels=3, channels=BASELINE_CHANNELS)

def make_3ch_dataset(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                    in_channels=3, augment=False)

def make_3ch_dataset_aug(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                    in_channels=3, augment=True)

print(f"Experiment B: 3-channel input (x, y, confidence)")
print(f"  Trainable params: {make_3ch_model().trainable_param_count():,}")

t0 = time.time()
results_B = run_rep_kfold_cv(
    model_factory=make_3ch_model,
    dataset_factory=make_3ch_dataset,
    rep_segments=reps_with_kps,
    n_splits=N_SPLITS,
    n_epochs=50,
    batch_size=16,
    lr=1e-3,
    patience=15,
    device_str=DEVICE,
    random_state=RANDOM_STATE,
    train_dataset_factory=make_3ch_dataset_aug,
)
time_B = time.time() - t0

accs_B = [f["val_accuracy"] for f in results_B["fold_results"]]
print(f"\nExperiment B results:")
print(f"  Per-fold: {[f'{a:.1%}' for a in accs_B]}")
print(f"  Mean: {np.mean(accs_B):.1%} +/- {np.std(accs_B):.1%}")
print(f"  Time: {time_B:.0f}s")

In [ ]:
# Compare A vs B
mean_A = np.mean(accs_A)
mean_B = np.mean(accs_B)

comparison_ab = pd.DataFrame({
    "Experiment": ["A: 2-channel (x,y)", "B: 3-channel (x,y,conf)"],
    "Mean Accuracy": [f"{mean_A:.1%}", f"{mean_B:.1%}"],
    "Std": [f"{np.std(accs_A):.1%}", f"{np.std(accs_B):.1%}"],
    "Time (s)": [f"{time_A:.0f}", f"{time_B:.0f}"],
})
print(comparison_ab.to_string(index=False))

if mean_B > mean_A:
    WINNER_IN_CH = 3
    winner_ch_name = "B: 3-channel"
    winner_ch_accs = accs_B
    winner_ch_results = results_B
    winner_ch_time = time_B
else:
    WINNER_IN_CH = 2
    winner_ch_name = "A: 2-channel"
    winner_ch_accs = accs_A
    winner_ch_results = results_A
    winner_ch_time = time_A

print(f"\nWinner: {winner_ch_name} ({np.mean(winner_ch_accs):.1%})")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["2-channel (x,y)", "3-channel (x,y,conf)"],
    [mean_A, mean_B],
    yerr=[np.std(accs_A), np.std(accs_B)],
    capsize=8, color=["#4C72B0", "#DD8452"], alpha=0.85, edgecolor="black",
)
for bar, m in zip(bars, [mean_A, mean_B]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{m:.1%}", ha="center", fontweight="bold")
ax.set_ylabel("Accuracy")
ax.set_title("Input Channels Comparison")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ab_channels.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Experiment C: Model Capacity

Compare three architectures using the winning input channel:
- **Small**: 2 blocks (32, 64)
- **Medium**: 3 blocks (64, 64, 128)
- **Large**: 4 blocks (64, 128, 128, 256)

In [ ]:
capacity_results = {}

for config_name in ["small", "medium", "large"]:
    channels = CONFIGS[config_name]

    def make_model(ch=channels):
        return PushUpSTGCN(in_channels=WINNER_IN_CH, channels=ch)

    def make_dataset(reps, in_ch=WINNER_IN_CH):
        return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                        in_channels=in_ch, augment=False)

    def make_dataset_aug(reps, in_ch=WINNER_IN_CH):
        return PushUpRepSkeletonDataset(reps, n_frames=BASELINE_N_FRAMES,
                                        in_channels=in_ch, augment=True)

    n_params = make_model().trainable_param_count()
    print(f"\n{'='*60}")
    print(f"{config_name.upper()}: {channels} — {n_params:,} params")
    print(f"{'='*60}")

    t0 = time.time()
    res = run_rep_kfold_cv(
        model_factory=make_model,
        dataset_factory=make_dataset,
        rep_segments=reps_with_kps,
        n_splits=N_SPLITS,
        n_epochs=50,
        batch_size=16,
        lr=1e-3,
        patience=15,
        device_str=DEVICE,
        random_state=RANDOM_STATE,
        train_dataset_factory=make_dataset_aug,
    )
    elapsed = time.time() - t0

    accs = [f["val_accuracy"] for f in res["fold_results"]]
    print(f"  Per-fold: {[f'{a:.1%}' for a in accs]}")
    print(f"  Mean: {np.mean(accs):.1%} +/- {np.std(accs):.1%}")
    print(f"  Time: {elapsed:.0f}s")

    capacity_results[config_name] = {
        "accs": accs,
        "results": res,
        "time": elapsed,
        "params": n_params,
        "channels": channels,
    }

In [ ]:
# Capacity comparison table and chart
cap_labels = ["small", "medium", "large"]
cap_display = [f"Small {CONFIGS['small']}", f"Medium {CONFIGS['medium']}",
               f"Large {CONFIGS['large']}"]

cap_df = pd.DataFrame({
    "Config": cap_display,
    "Params": [f"{capacity_results[k]['params']:,}" for k in cap_labels],
    "Mean Accuracy": [f"{np.mean(capacity_results[k]['accs']):.1%}" for k in cap_labels],
    "Std": [f"{np.std(capacity_results[k]['accs']):.1%}" for k in cap_labels],
    "Time (s)": [f"{capacity_results[k]['time']:.0f}" for k in cap_labels],
})
print(cap_df.to_string(index=False))

means_c = [np.mean(capacity_results[k]["accs"]) for k in cap_labels]
stds_c = [np.std(capacity_results[k]["accs"]) for k in cap_labels]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#4C72B0", "#55A868", "#C44E52"]
bars = ax.bar(cap_display, means_c, yerr=stds_c, capsize=8,
              color=colors, alpha=0.85, edgecolor="black")
for bar, m, s in zip(bars, means_c, stds_c):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
            f"{m:.1%}", ha="center", fontweight="bold")
ax.set_ylabel("Accuracy")
ax.set_title(f"Model Capacity Comparison ({WINNER_IN_CH}-channel input)")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "capacity_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best_capacity = max(cap_labels, key=lambda k: np.mean(capacity_results[k]["accs"]))
WINNER_CHANNELS = CONFIGS[best_capacity]
print(f"\nBest capacity: {best_capacity} ({np.mean(capacity_results[best_capacity]['accs']):.1%})")

---
## Experiment D: Temporal Sampling

How many frames per rep? Compare 16, 32, and 64 frames
using the winning input channels and model capacity.

In [ ]:
temporal_results = {}

for n_frames in [16, 32, 64]:
    def make_model(ch=WINNER_CHANNELS):
        return PushUpSTGCN(in_channels=WINNER_IN_CH, channels=ch)

    def make_dataset(reps, nf=n_frames):
        return PushUpRepSkeletonDataset(reps, n_frames=nf,
                                        in_channels=WINNER_IN_CH, augment=False)

    def make_dataset_aug(reps, nf=n_frames):
        return PushUpRepSkeletonDataset(reps, n_frames=nf,
                                        in_channels=WINNER_IN_CH, augment=True)

    print(f"\n--- n_frames={n_frames} ---")

    t0 = time.time()
    res = run_rep_kfold_cv(
        model_factory=make_model,
        dataset_factory=make_dataset,
        rep_segments=reps_with_kps,
        n_splits=N_SPLITS,
        n_epochs=50,
        batch_size=16,
        lr=1e-3,
        patience=15,
        device_str=DEVICE,
        random_state=RANDOM_STATE,
        train_dataset_factory=make_dataset_aug,
    )
    elapsed = time.time() - t0

    accs = [f["val_accuracy"] for f in res["fold_results"]]
    print(f"  Mean: {np.mean(accs):.1%} +/- {np.std(accs):.1%} ({elapsed:.0f}s)")

    temporal_results[n_frames] = {
        "accs": accs,
        "results": res,
        "time": elapsed,
    }

In [ ]:
# Temporal comparison
temp_labels = [16, 32, 64]

temp_df = pd.DataFrame({
    "Frames": temp_labels,
    "Mean Accuracy": [f"{np.mean(temporal_results[k]['accs']):.1%}" for k in temp_labels],
    "Std": [f"{np.std(temporal_results[k]['accs']):.1%}" for k in temp_labels],
    "Time (s)": [f"{temporal_results[k]['time']:.0f}" for k in temp_labels],
})
print(temp_df.to_string(index=False))

means_t = [np.mean(temporal_results[k]["accs"]) for k in temp_labels]
stds_t = [np.std(temporal_results[k]["accs"]) for k in temp_labels]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(n) for n in temp_labels], means_t, yerr=stds_t, capsize=8,
       color=["#4C72B0", "#55A868", "#C44E52"], alpha=0.85, edgecolor="black")
for i, (m, s) in enumerate(zip(means_t, stds_t)):
    ax.text(i, m + s + 0.01, f"{m:.1%}", ha="center", fontweight="bold")
ax.set_xlabel("Frames per rep")
ax.set_ylabel("Accuracy")
ax.set_title("Temporal Sampling Comparison")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "temporal_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

WINNER_N_FRAMES = max(temp_labels, key=lambda k: np.mean(temporal_results[k]["accs"]))
print(f"\nBest temporal sampling: {WINNER_N_FRAMES} frames "
      f"({np.mean(temporal_results[WINNER_N_FRAMES]['accs']):.1%})")

---
## Experiment E: Hyperparameter Tuning

Grid search over learning rate and batch size using the best config from above.

In [ ]:
def _make_best_model():
    return PushUpSTGCN(in_channels=WINNER_IN_CH, channels=WINNER_CHANNELS)

def _make_best_dataset(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=WINNER_N_FRAMES,
                                    in_channels=WINNER_IN_CH, augment=False)

def _make_best_dataset_aug(reps):
    return PushUpRepSkeletonDataset(reps, n_frames=WINNER_N_FRAMES,
                                    in_channels=WINNER_IN_CH, augment=True)

print(f"Tuning config: {best_capacity} capacity, {WINNER_IN_CH}-channel, {WINNER_N_FRAMES} frames")
print(f"  Trainable params: {_make_best_model().trainable_param_count():,}")

LR_OPTIONS = [5e-4, 1e-3, 5e-3]
BS_OPTIONS = [8, 16, 32]

hp_results = []

for lr in LR_OPTIONS:
    for bs in BS_OPTIONS:
        print(f"\n--- lr={lr}, batch_size={bs} ---")
        t0 = time.time()
        res = run_rep_kfold_cv(
            model_factory=_make_best_model,
            dataset_factory=_make_best_dataset,
            rep_segments=reps_with_kps,
            n_splits=N_SPLITS,
            n_epochs=50,
            batch_size=bs,
            lr=lr,
            patience=15,
            device_str=DEVICE,
            random_state=RANDOM_STATE,
            train_dataset_factory=_make_best_dataset_aug,
        )
        elapsed = time.time() - t0
        accs = [f["val_accuracy"] for f in res["fold_results"]]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        print(f"  Mean: {mean_acc:.1%} +/- {std_acc:.1%} ({elapsed:.0f}s)")

        hp_results.append({
            "lr": lr,
            "batch_size": bs,
            "mean_acc": mean_acc,
            "std_acc": std_acc,
            "per_fold": accs,
            "time": elapsed,
            "results": res,
        })

In [ ]:
# HP results table + heatmap
hp_df = pd.DataFrame([
    {
        "LR": f"{r['lr']:.0e}",
        "Batch Size": r["batch_size"],
        "Mean Accuracy": f"{r['mean_acc']:.1%}",
        "Std": f"{r['std_acc']:.1%}",
        "Time (s)": f"{r['time']:.0f}",
    }
    for r in hp_results
])
print(hp_df.to_string(index=False))

pivot = pd.DataFrame(hp_results).pivot(
    index="batch_size", columns="lr", values="mean_acc"
)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:.0e}" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Learning Rate")
ax.set_ylabel("Batch Size")
ax.set_title("Hyperparameter Grid Search")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.1%}", ha="center", va="center",
                fontweight="bold", color="white" if val > 0.75 else "black")

plt.colorbar(im, label="Accuracy")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hp_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

best_hp = max(hp_results, key=lambda r: r["mean_acc"])
print(f"\nBest hyperparams: lr={best_hp['lr']:.0e}, batch_size={best_hp['batch_size']}")
print(f"  Accuracy: {best_hp['mean_acc']:.1%} +/- {best_hp['std_acc']:.1%}")

---
## Final Comparison

Summary of all experiments.

In [ ]:
all_experiments = []

# A: Baseline
all_experiments.append({
    "Experiment": "A: Baseline (medium, 2ch, 32f)",
    "Channels": str(CONFIGS["medium"]),
    "Input Ch": 2,
    "Frames": 32,
    "Params": f"{make_baseline_model().trainable_param_count():,}",
    "Mean Acc": np.mean(accs_A),
    "Std": np.std(accs_A),
})

# B: 3-channel (if different from A)
if WINNER_IN_CH == 3:
    all_experiments.append({
        "Experiment": "B: 3-channel (x,y,conf)",
        "Channels": str(CONFIGS["medium"]),
        "Input Ch": 3,
        "Frames": 32,
        "Params": f"{make_3ch_model().trainable_param_count():,}",
        "Mean Acc": np.mean(accs_B),
        "Std": np.std(accs_B),
    })

# C: Capacity variants
for config_name in ["small", "medium", "large"]:
    r = capacity_results[config_name]
    all_experiments.append({
        "Experiment": f"C: {config_name.title()} {CONFIGS[config_name]}",
        "Channels": str(CONFIGS[config_name]),
        "Input Ch": WINNER_IN_CH,
        "Frames": 32,
        "Params": f"{r['params']:,}",
        "Mean Acc": np.mean(r["accs"]),
        "Std": np.std(r["accs"]),
    })

# D: Temporal variants
for nf in [16, 32, 64]:
    r = temporal_results[nf]
    all_experiments.append({
        "Experiment": f"D: {nf} frames",
        "Channels": str(WINNER_CHANNELS),
        "Input Ch": WINNER_IN_CH,
        "Frames": nf,
        "Params": f"{_make_best_model().trainable_param_count():,}",
        "Mean Acc": np.mean(r["accs"]),
        "Std": np.std(r["accs"]),
    })

# E: Best HP
all_experiments.append({
    "Experiment": f"E: Best HP (lr={best_hp['lr']:.0e}, bs={best_hp['batch_size']})",
    "Channels": str(WINNER_CHANNELS),
    "Input Ch": WINNER_IN_CH,
    "Frames": WINNER_N_FRAMES,
    "Params": f"{_make_best_model().trainable_param_count():,}",
    "Mean Acc": best_hp["mean_acc"],
    "Std": best_hp["std_acc"],
})

final_df = pd.DataFrame(all_experiments)
print(final_df.to_string(index=False))

In [ ]:
# Final bar chart
names = final_df["Experiment"].tolist()
means_f = final_df["Mean Acc"].tolist()
stds_f = final_df["Std"].tolist()

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
bars = ax.bar(range(len(names)), means_f, yerr=stds_f, capsize=5,
              color=colors, alpha=0.85, edgecolor="black")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=35, ha="right", fontsize=8)

for bar, m, s in zip(bars, means_f, stds_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
            f"{m:.1%}", ha="center", fontweight="bold", fontsize=8)

ax.set_ylabel("Accuracy")
ax.set_title(f"ST-GCN Study: All Configurations ({len(reps_with_kps)} reps, {N_SPLITS}-fold CV)")
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best_row = final_df.loc[final_df["Mean Acc"].idxmax()]
print(f"\nBest overall: {best_row['Experiment']}")
print(f"  Accuracy: {best_row['Mean Acc']:.1%} +/- {best_row['Std']:.1%}")

---
## Overfitting Analysis

1. Train vs val loss curves
2. Confusion matrices
3. Per-video error analysis

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# ---- 1. Train vs Val Loss Curves ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs_to_plot = [
    ("Baseline (A)", results_A),
    ("Best HP (E)", best_hp["results"]),
]

for ax, (name, res) in zip(axes, configs_to_plot):
    for fold_res in res["fold_results"]:
        if "epoch_train_losses" not in fold_res:
            continue
        epochs = range(1, len(fold_res["epoch_train_losses"]) + 1)
        ax.plot(epochs, fold_res["epoch_train_losses"], "b-", alpha=0.3)
        ax.plot(epochs, fold_res["epoch_val_losses"], "r-", alpha=0.3)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(name)
    ax.legend(["Train", "Val"], loc="upper right")

plt.suptitle("Train vs Val Loss (blue=train, red=val)", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- 2. Confusion Matrices ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

configs_cm = [
    ("Baseline (A)", results_A),
    ("Best HP (E)", best_hp["results"]),
]

for ax, (name, res) in zip(axes, configs_cm):
    preds = res["per_rep_preds"]
    true = res["per_rep_true"]
    valid = [(p, t) for p, t in zip(preds, true) if p is not None]
    if not valid:
        continue
    p, t = zip(*valid)
    cm = confusion_matrix(t, p, labels=[0, 1])

    im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Good", "Bad"])
    ax.set_yticklabels(["Good", "Bad"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(name)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=16, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.suptitle("Confusion Matrices — per-rep CV predictions", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

for name, res in configs_cm:
    preds = [p for p in res["per_rep_preds"] if p is not None]
    true = [t for p, t in zip(res["per_rep_preds"], res["per_rep_true"]) if p is not None]
    print(f"\n{name}:")
    print(classification_report(true, preds, target_names=["Good", "Bad"]))

In [ ]:
# ---- 3. Per-Video Error Analysis (Best HP model) ----
best_preds = best_hp["results"]["per_rep_preds"]
best_true = best_hp["results"]["per_rep_true"]

video_stats = {}
for i, rep in enumerate(reps_with_kps):
    vid = rep["video_id"]
    pred = best_preds[i]
    true = best_true[i]
    if pred is None:
        continue
    if vid not in video_stats:
        video_stats[vid] = {"correct": 0, "wrong": 0, "label": rep["label"]}
    if pred == true:
        video_stats[vid]["correct"] += 1
    else:
        video_stats[vid]["wrong"] += 1

error_videos = {k: v for k, v in video_stats.items() if v["wrong"] > 0}
perfect_videos = {k: v for k, v in video_stats.items() if v["wrong"] == 0}

print(f"Total videos: {len(video_stats)}")
print(f"Perfect (0 errors): {len(perfect_videos)}")
print(f"With errors: {len(error_videos)}")

if error_videos:
    print(f"\nVideos with misclassified reps:")
    print(f"{'Video':<35} {'Label':<8} {'Correct':<10} {'Wrong':<8}")
    print("-" * 65)
    for vid, stats in sorted(error_videos.items(), key=lambda x: -x[1]["wrong"]):
        label = "Good" if stats["label"] == 0 else "Bad"
        print(f"{vid:<35} {label:<8} {stats['correct']:<10} {stats['wrong']:<8}")

good_errors = sum(v["wrong"] for v in error_videos.values() if v["label"] == 0)
bad_errors = sum(v["wrong"] for v in error_videos.values() if v["label"] == 1)
print(f"\nMisclassified reps from 'good' videos: {good_errors}")
print(f"Misclassified reps from 'bad' videos: {bad_errors}")

plt.figure(figsize=(8, 4))
error_counts = [v["wrong"] for v in video_stats.values()]
plt.hist(error_counts, bins=range(max(error_counts) + 2), edgecolor="black", alpha=0.7)
plt.xlabel("Number of misclassified reps per video")
plt.ylabel("Number of videos")
plt.title("Error Distribution Across Videos (Best HP)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Save Results

In [ ]:
# Save summary CSV
final_df.to_csv(OUTPUT_DIR / "stgcn_study_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'stgcn_study_results.csv'}")

# Save HP grid CSV
hp_df.to_csv(OUTPUT_DIR / "stgcn_hp_grid.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'stgcn_hp_grid.csv'}")

# Save best model weights + config
if best_hp["results"]["best_state"] is not None:
    torch.save({
        "state_dict": best_hp["results"]["best_state"],
        "in_channels": WINNER_IN_CH,
        "channels": WINNER_CHANNELS,
        "n_frames": WINNER_N_FRAMES,
        "accuracy": best_hp["mean_acc"],
        "lr": best_hp["lr"],
        "batch_size": best_hp["batch_size"],
        "dropout": 0.2,
    }, OUTPUT_DIR / "stgcn_best.pt")
    print(f"Saved: {OUTPUT_DIR / 'stgcn_best.pt'}")

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")
print("\nFigures:")
for f in sorted(OUTPUT_DIR.glob("*.png")):
    print(f"  {f.name}")
print("CSVs:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

---
## Inference Pipelines

Two modes for counting good push-up reps:

1. **Automatic** (`infer_automatic`): State machine auto-detects reps from pre-recorded video, classifies using keypoints only.
2. **Real-time** (`live_demo.py`): Webcam with instant per-rep classification as reps complete. No recording delay.

Key advantage: classification uses skeleton keypoints only — no video frame decoding, no domain gap.

In [ ]:
from inference import load_model, infer_automatic, summarize, print_results

# Load the best saved model
device = DEVICE
model, config = load_model("outputs/stgcn_best.pt", device=device)
print(f"Using device: {device}")
print(f"Input channels: {config.get('in_channels')}")
print(f"Frames per rep: {config.get('n_frames')}")
print(f"CV accuracy: {config.get('accuracy', 0):.1%}")

In [ ]:
# Automatic rep detection (no annotation needed)
# Pipeline: YOLO keypoints -> State Machine -> ST-GCN classification
#
# Classification uses keypoints only — no video frame decoding!

auto_test_videos = [
    # "videos/my_test_video.mp4",
]

if not auto_test_videos:
    print("No test videos defined.")
    print("Add video paths to auto_test_videos above, then re-run this cell.")
else:
    all_auto_results = []

    for video_path in auto_test_videos:
        results_auto = infer_automatic(
            video_path=video_path,
            model=model,
            config=config,
            keypoint_dir=KEYPOINT_DIR,
            device=device,
        )
        print_results(results_auto, f"Automatic — {Path(video_path).name}")

        s = summarize(results_auto)
        print(f"\n  GOOD reps: {s['good_reps']} / {s['total_reps']}")

        all_auto_results.extend(results_auto)

In [ ]:
# Live webcam demo (real-time per-rep classification)
# Run from terminal:
#   cd stgcn_study
#   python live_demo.py
#   python live_demo.py --camera 1       # external webcam
#   python live_demo.py --no-skeleton    # hide skeleton overlay
#
# Key advantage over R3D:
#   - Classifies from keypoints only (no video crops)
#   - No domain gap between training and live inference
#   - Instant feedback per rep (no recording/processing delay)

print("Live demo must be run from terminal:")
print("  cd stgcn_study")
print("  python live_demo.py")
print()
print("Controls: Q/ESC = quit, R = reset counters")
print("Each rep is classified instantly when completed.")

### Inference Pipeline Summary

| | Automatic (`infer_automatic`) | Real-time (`live_demo.py`) |
|---|---|---|
| **Rep detection** | State machine (offline) | State machine (real-time) |
| **Classification input** | Keypoints only | Keypoints only |
| **Pipeline** | YOLO kps -> State Machine -> ST-GCN | Webcam -> YOLO -> State Machine -> ST-GCN |
| **Latency** | Batch (offline) | Instant per-rep |
| **Domain gap** | None | None |

Unlike R3D (which needs video frame crops), ST-GCN classifies from joint coordinates only.
This eliminates the domain gap that caused R3D's live demo to fail.